# replace-final-head — worked example 1: Swap the Final Classifier by Reading in_features from the Old Head

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `replace-final-head`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In transfer learning, you replace a pretrained model's final classification layer with a new `nn.Linear` that outputs the target number of classes. The key is to read `model.fc.in_features` from the old head — this gives the backbone's output width, which the new head must match. Assigning `model.fc = nn.Linear(in_features, new_num_classes)` registers the new head as a proper child module.

## Worked solution

**Step 1 — read `in_features` from the old head.** `model.fc.in_features` tells us the backbone output width. We must NOT hardcode this — it must be read from the existing layer so the function works for any backbone.

**Step 2 — create a new `nn.Linear`.** `nn.Linear(in_features, new_num_classes)` creates a fresh layer with random weights and `requires_grad=True`. It knows nothing about the old 1000-class weights.

**Step 3 — assign to `model.fc`.** Simple attribute assignment on a `nn.Module` replaces the old submodule completely. The old head is removed from `model.parameters()` and the new one takes its place.

**Step 4 — verify the swap.** Check `model.fc.out_features == new_num_classes` and that `model.fc.in_features` still equals the backbone width.

**Step 5 — test forward pass.** Run a batch through the model and verify output shape is `(batch_size, new_num_classes)`.

In [ ]:
import torch as t
import torch.nn as nn

class ToyPretrained(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 16)
        )
        self.fc = nn.Linear(16, 1000)  # pretrained 1000-class head
    def forward(self, x):
        return self.fc(self.backbone(x))

def replace_head(model: nn.Module, new_num_classes: int) -> nn.Module:
    in_features = model.fc.in_features          # read backbone width from old head
    model.fc = nn.Linear(in_features, new_num_classes)  # replace
    return model

# --- exercise and print ---
t.manual_seed(0)
model = ToyPretrained()
print('Before swap:')
print('  fc.out_features:', model.fc.out_features)  # 1000

model = replace_head(model, new_num_classes=10)
print('After swap:')
print('  fc.out_features:', model.fc.out_features)  # 10
print('  fc.in_features: ', model.fc.in_features)   # 16 (unchanged)
print('  fc.weight.requires_grad:', model.fc.weight.requires_grad)  # True

# Forward pass check
x = t.randn(4, 10)
out = model(x)
print('  output shape:   ', tuple(out.shape))  # (4, 10)